# Bao cao Tuan 1 (W1-02) - Baseline & Evaluation

Notebook nay kiem tra PyTorch/CPU/GPU thuc te, nap class map dung chung va xac minh evaluation pipeline tren du lieu synthetic. Notebook khong huan luyen mo hinh va khong doc tap Test.

In [1]:
from pathlib import Path
import platform
import sys
import time

import torch

def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'data/processed/v1/class_map.json').is_file():
            return candidate
    raise FileNotFoundError('Khong tim thay data/processed/v1/class_map.json')

ROOT = find_repository_root(Path.cwd().resolve())
REPORT_DIR = ROOT / 'reports' / 'tv2'
REPORT_DIR.mkdir(parents=True, exist_ok=True)

cuda_available = torch.cuda.is_available()
device = torch.device('cuda' if cuda_available else 'cpu')
device_name = torch.cuda.get_device_name(0) if cuda_available else platform.processor() or 'CPU'

matrix_a = torch.rand((256, 256), device=device)
matrix_b = torch.rand((256, 256), device=device)
started = time.perf_counter()
_ = matrix_a @ matrix_b
if cuda_available:
    torch.cuda.synchronize()
elapsed_ms = (time.perf_counter() - started) * 1000

environment_lines = [
    f'Python: {platform.python_version()}',
    f'Platform: {platform.platform()}',
    f'PyTorch: {torch.__version__}',
    f'CUDA available: {cuda_available}',
    f'Compute device: {device.type.upper()}',
    f'Device name: {device_name}',
    f'256x256 matmul smoke test: PASS ({elapsed_ms:.3f} ms)',
]
if cuda_available:
    props = torch.cuda.get_device_properties(0)
    environment_lines.extend([
        f'CUDA runtime: {torch.version.cuda or "N/A"}',
        f'GPU count: {torch.cuda.device_count()}',
        f'GPU memory: {props.total_memory / (1024 ** 3):.2f} GB',
    ])

environment_report = '\n'.join(environment_lines) + '\n'
environment_path = REPORT_DIR / 'environment_report.txt'
environment_path.write_text(environment_report, encoding='utf-8')
print(environment_report)
print(f'Report: {environment_path.relative_to(ROOT)}')

Python: 3.12.14
Platform: Windows-11-10.0.26200-SP0
PyTorch: 2.8.0+cpu
CUDA available: False
Compute device: CPU
Device name: Intel64 Family 6 Model 186 Stepping 2, GenuineIntel
256x256 matmul smoke test: PASS (1.707 ms)

Report: reports\tv2\environment_report.txt


In [2]:
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.metrics import compute_metrics, load_class_map, plot_confusion_matrix

CLASS_MAP_PATH = ROOT / 'data' / 'processed' / 'v1' / 'class_map.json'
labels, target_names = load_class_map(CLASS_MAP_PATH)
print(f'Loaded {len(labels)} classes')
print(dict(zip(labels, target_names)))

Loaded 8 classes
{0: 'large_antlered_muntjac', 1: 'annamite_striped_rabbit', 2: 'sambar', 3: 'chinese_serow', 4: 'common_palm_civet', 5: 'masked_palm_civet', 6: 'silver_pheasant', 7: 'eurasian_wild_pig'}


In [3]:
# Day la synthetic/mock data dung de kiem tra evaluation pipeline,
# khong phai ket qua huan luyen mo hinh that.
import matplotlib.pyplot as plt
import numpy as np

y_true = np.array([0, 0, 1, 1, 2, 3, 4, 5, 6, 7], dtype=int)
y_pred = np.array([0, 1, 1, 1, 2, 3, 4, 5, 7, 7], dtype=int)
metrics = compute_metrics(y_true, y_pred, CLASS_MAP_PATH)

print('SYNTHETIC EVALUATION - NOT A TRAINED MODEL RESULT')
for metric_name in ('accuracy', 'macro_precision', 'macro_recall', 'macro_f1'):
    print(f'{metric_name}: {metrics[metric_name]:.6f}')
print(metrics['classification_report_str'])

figure_path = REPORT_DIR / 'demo_confusion_matrix.png'
fig, matrix = plot_confusion_matrix(
    y_true,
    y_pred,
    output_path=figure_path,
    class_map_path=CLASS_MAP_PATH,
    title='Synthetic evaluation confusion matrix',
)
assert matrix.shape == (len(labels), len(labels))
assert np.isfinite([metrics['macro_precision'], metrics['macro_recall'], metrics['macro_f1']]).all()

perfect = compute_metrics(labels, labels, CLASS_MAP_PATH)
assert perfect['accuracy'] == 1.0 and perfect['macro_f1'] == 1.0
single = compute_metrics([labels[0]], [labels[0]], CLASS_MAP_PATH)
assert single['accuracy'] == 1.0
assert np.isclose(single['macro_f1'], 1 / len(labels))
assert single['confusion_matrix'].shape == (len(labels), len(labels))
missing = compute_metrics([labels[0], labels[1]], [labels[0], labels[0]], CLASS_MAP_PATH)
assert missing['classification_report'][target_names[1]]['recall'] == 0.0
assert missing['classification_report'][target_names[1]]['f1-score'] == 0.0
for true, pred in (([], []), ([labels[0], labels[1]], [labels[0]]), ([labels[0], 99], [labels[0], labels[0]])):
    try:
        compute_metrics(true, pred, CLASS_MAP_PATH)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid metric input was accepted')
print('Metric edge-case assertions: PASS')
print(f'Confusion matrix shape: {matrix.shape}')
print(f'Figure: {figure_path.relative_to(ROOT)}')
plt.close(fig)

SYNTHETIC EVALUATION - NOT A TRAINED MODEL RESULT
accuracy: 0.800000
macro_precision: 0.770833
macro_recall: 0.812500
macro_f1: 0.766667
                         precision    recall  f1-score   support

 large_antlered_muntjac       1.00      0.50      0.67         2
annamite_striped_rabbit       0.67      1.00      0.80         2
                 sambar       1.00      1.00      1.00         1
          chinese_serow       1.00      1.00      1.00         1
      common_palm_civet       1.00      1.00      1.00         1
      masked_palm_civet       1.00      1.00      1.00         1
        silver_pheasant       0.00      0.00      0.00         1
      eurasian_wild_pig       0.50      1.00      0.67         1

               accuracy                           0.80        10
              macro avg       0.77      0.81      0.77        10
           weighted avg       0.78      0.80      0.76        10



Metric edge-case assertions: PASS
Confusion matrix shape: (8, 8)
Figure: reports\tv2\demo_confusion_matrix.png


In [4]:
print('W1-02 validation summary')
print(f'- Runtime device detected in this run: {device.type.upper()}')
print('- Checkpoint and early stopping metric: Validation Macro-F1')
print('- Test isolation: do not load, inspect, or tune on Test before the locked Week 4 evaluation')
print('- B1/B2: use the same eligible image list and the same site-level train/val/test manifest')
print('- This notebook used synthetic data only; no real model training result is claimed')

W1-02 validation summary
- Runtime device detected in this run: CPU
- Checkpoint and early stopping metric: Validation Macro-F1
- Test isolation: do not load, inspect, or tune on Test before the locked Week 4 evaluation
- B1/B2: use the same eligible image list and the same site-level train/val/test manifest
- This notebook used synthetic data only; no real model training result is claimed
